# Week 2 Simulator Results | QNN Fourier Spectrum

**Project 9 · QML**  
**Student:** Rana Muhammad Hamza · **Roll number:** 25280103  
**Milestone:** Week 2 progress review, results draft, simulator code

## Purpose

This notebook completes the Week 2 simulator study from the project synopsis. It combines the saved one-dimensional L=1–6 results with a matched-parameter two-dimensional serial/parallel comparison, a fixed-encoding depth ablation, an entangler ablation, a two-spiral classification benchmark, and a gate-level depolarizing-noise / finite-shot sweep.

The project uses a noiseless statevector for training and spectrum analysis, then Qiskit Aer for shot-based noisy inference. It does not claim hardware execution. The data, fixed folds, run settings, trained parameters, figures, and machine-readable result tables are saved under `outputs/week2/`.

## Research question and preregistered checks

**Question.** How do encoding repetitions and layer count set the frequencies a QNN can represent, and how does serial versus parallel encoding change learned coefficients and resources?

**Hypothesis.** Repeating Pauli-rotation encodings expands the candidate Fourier support. Each feature's coordinate-wise integer frequency is bounded by its number of encodings; ansatz structure and parameters can suppress allowed modes. The 2D circuit should learn the target mixed mode `(1, -1)` while keeping active modes inside `|k1|, |k2| ≤ L`.

**Falsifier.** A coefficient beyond the declared support must exceed `max(1e-6, 0.01*c_max)` and persist at both 64×64 and 128×128 grids, across independent initializations. The notebook reports candidate support and measured coefficients separately.

## Run order

Run this notebook from the project root with the **Python (QNN Fourier Spectrum)** kernel. It reads the seeded Week 1 datasets and fixed folds, and reuses the completed Week 1 Advancement one-dimensional sweep. It then runs all Week 2 simulations and writes its results into the dedicated `outputs/week2/` folder.


In [1]:
from pathlib import Path
import json
import math
import platform
import time
import warnings
import importlib.metadata as metadata
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "outputs/week1/week1_inputs_and_fixed_folds.npz").exists():
    ROOT = ROOT.parent
if not (ROOT / "outputs/week1/week1_inputs_and_fixed_folds.npz").exists():
    raise FileNotFoundError("Open this notebook from the qnn-fourier-spectrum project.")

OUT = ROOT / "outputs" / "week2"
FIG = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

SEED = 42
INIT_SEEDS = tuple(range(5))
LAYERS = tuple(range(1, 7))
ADAM_STEPS = 1000
ADAM_RATE = 0.01
SPECTRUM_ABS_TOL = 1e-6
SPECTRUM_REL_TOL = 0.01
NOISE_LEVELS = (0.0, 0.001, 0.005, 0.01)
NOISE_REPEATS = 5
NOISE_SHOTS = 1024
NOISE_POINTS = 256

print("Project root:", ROOT)
print("Output folder:", OUT)
print("Python:", platform.python_version())
for package in ("numpy", "pandas", "scikit-learn", "qiskit", "qiskit-aer"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: version metadata unavailable")


Project root: C:\Users\Teacher\Desktop\qml\qnn-fourier-spectrum
Output folder: C:\Users\Teacher\Desktop\qml\qnn-fourier-spectrum\outputs\week2
Python: 3.13.7
numpy: 2.5.5
pandas: 2.3.3
scikit-learn: 1.9.1
qiskit: 2.6.2
qiskit-aer: 0.18.2


## 1. Seeded data and prior 1D evidence

The regression and spiral inputs are loaded from the Week 1 seed-42 bundle. The five stratified test folds are reused unchanged. This notebook also incorporates the already-run Week 1 Advancement L=1–6 results so the 1D and 2D experiments appear together without repeating that completed 1D training job.


In [2]:
data_path = ROOT / "outputs" / "week1" / "week1_inputs_and_fixed_folds.npz"
data = np.load(data_path)
X1_train = data["X_1d_train"]
y1_train = data["y_1d_train"]
X1_grid = data["X_1d_grid"]
y1_grid = data["y_1d_grid"]
X2_train = data["X_2d_train"]
y2_train = data["y_2d_train"]
X2_grid64 = data["X_2d_grid"]
y2_grid64 = data["y_2d_grid"]
X_spiral = data["X_spiral"]
y_spiral = data["y_spiral"]
fold_train = data["fold_train"]
fold_test = data["fold_test"]

def target_2d(X):
    X = np.asarray(X, dtype=float)
    x1, x2 = X[:, 0], X[:, 1]
    return 0.4 * np.sin(x1) + 0.3 * np.cos(2 * x2) + 0.3 * np.sin(x1 - x2)

axis128 = np.linspace(-np.pi, np.pi, 128, endpoint=False)
grid_x1_128, grid_x2_128 = np.meshgrid(axis128, axis128, indexing="xy")
X2_grid128 = np.column_stack((grid_x1_128.ravel(), grid_x2_128.ravel()))
y2_grid128 = target_2d(X2_grid128).reshape(128, 128)

assert X1_train.shape == (128, 1)
assert X2_train.shape == (2048, 2)
assert X_spiral.shape == (2000, 2)
assert fold_train.shape == (5, 1600) and fold_test.shape == (5, 400)
coverage = np.zeros(len(y_spiral), dtype=int)
for tr, te in zip(fold_train, fold_test):
    assert len(np.intersect1d(tr, te)) == 0
    coverage[te] += 1
assert np.all(coverage == 1)

advancement_dir = ROOT / "outputs" / "week1Advancement"
one_d_sweep = pd.read_csv(advancement_dir / "advancement_layer_sweep.csv")
one_d_summary = pd.read_csv(advancement_dir / "advancement_layer_summary.csv")
assert set(one_d_sweep["layers"].astype(int)) == set(LAYERS)
assert set(one_d_sweep["target"].unique()) == {"week1_original", "high_harmonic"}
assert one_d_sweep.groupby(["target", "layers"]).size().eq(5).all()

print("1D train/grid:", X1_train.shape, X1_grid.shape)
print("2D train/grid:", X2_train.shape, y2_grid64.shape, "and refined", y2_grid128.shape)
print("Spiral examples/classes:", X_spiral.shape, np.bincount(y_spiral))
print("Reused fixed folds:", fold_train.shape, fold_test.shape)
print("Reused 1D Advancement fits:", len(one_d_sweep), "(two targets × six depths × five seeds)")
display(one_d_summary)


1D train/grid: (128, 1) (256, 1)
2D train/grid: (2048, 2) (64, 64) and refined (128, 128)
Spiral examples/classes: (2000, 2) [1000 1000]
Reused fixed folds: (5, 1600) (5, 400)
Reused 1D Advancement fits: 60 (two targets × six depths × five seeds)


,target,layers,parameters,qnn_grid_mse_mean,qnn_grid_mse_sd,train_mse_mean,fourier_KL_mse,outside_modes_256_total,outside_modes_512_total
0,high_harmonic,1,2,0.051271,1.321226e-12,0.053399,5.127122e-02,0,0
1,high_harmonic,2,4,0.118468,2.441850e-03,0.110755,5.337374e-02,0,0
2,high_harmonic,3,6,0.081278,5.538888e-02,0.067763,5.536454e-02,0,0
3,high_harmonic,4,8,0.043101,8.394318e-03,0.041927,2.050294e-02,0,0
4,high_harmonic,5,10,0.032155,8.963217e-03,0.027440,2.096070e-02,0,0
5,high_harmonic,6,12,0.010024,7.835521e-03,0.009261,1.575050e-31,0,0
6,week1_original,1,2,0.065933,5.326006e-11,0.051935,6.593298e-02,0,0
7,week1_original,2,4,0.146567,3.735180e-04,0.152484,2.093440e-02,0,0
8,week1_original,3,6,0.014984,1.259435e-11,0.013584,2.760833e-31,0,0
9,week1_original,4,8,0.008039,7.573686e-03,0.006400,3.480835e-31,0,0


## 2. Exact statevector model and gradient

Serial encoding uses one qubit and applies `RX(x1)`, `RZ(x2)`, then two trainable rotations per repetition. Parallel encoding applies those data rotations to separate qubits, uses the same two trainable parameters per repetition, and adds one CNOT. Both layouts therefore have `2L` trainable parameters. The circuit returns the first-qubit `Z` expectation.

Gradients use the exact statevector adjoint method. A small finite-difference check below verifies the implementation before the full sweep. Noise is applied later at inference through an explicit Aer depolarizing channel.


In [3]:
def build_schedule(layout, layers, entangler=True, deep_fixed3=False):
    if deep_fixed3:
        if layout != "parallel" or layers != 3:
            raise ValueError("The depth control is defined for the 3-encoding parallel circuit.")
        schedule = []
        for _encoding in range(3):
            schedule.extend([
                ("rx", "x1", 0, None),
                ("rz", "x2", 1, None),
            ])
            for _block in range(2):
                p = 2 * _encoding + _block
                schedule.extend([
                    ("ry", "param", 0, p),
                    ("rz", "param", 1, p),
                ])
                if entangler:
                    schedule.append(("cx", None, None, None))
        return 2, schedule, 6

    if layout not in {"serial", "parallel", "parallel_no_entangler"}:
        raise ValueError(f"Unknown layout: {layout}")
    n_qubits = 1 if layout == "serial" else 2
    use_entangler = layout == "parallel" and entangler
    schedule = []
    for layer in range(layers):
        if layout == "serial":
            schedule.extend([
                ("rx", "x1", 0, None),
                ("rz", "x2", 0, None),
                ("ry", "param", 0, 2 * layer),
                ("rz", "param", 0, 2 * layer + 1),
            ])
        else:
            schedule.extend([
                ("rx", "x1", 0, None),
                ("rz", "x2", 1, None),
                ("ry", "param", 0, 2 * layer),
                ("rz", "param", 1, 2 * layer + 1),
            ])
            if use_entangler:
                schedule.append(("cx", None, None, None))
    return n_qubits, schedule, 2 * layers


@lru_cache(maxsize=4)
def basis_pairs(n_qubits, qubit):
    mask = 1 << (n_qubits - 1 - qubit)
    zero = np.asarray([i for i in range(2 ** n_qubits) if not (i & mask)], dtype=int)
    one = zero | mask
    return zero, one


def apply_rotation(states, axis, angle, qubit, inverse=False):
    n_qubits = int(round(math.log2(states.shape[1])))
    zero, one = basis_pairs(n_qubits, qubit)
    theta = np.asarray(angle, dtype=float) * (-1.0 if inverse else 1.0)
    if theta.ndim == 0:
        c = np.cos(theta / 2)
        s = np.sin(theta / 2)
        phase_m = np.exp(-0.5j * theta)
        phase_p = np.exp(0.5j * theta)
    else:
        theta = theta[:, None]
        c = np.cos(theta / 2)
        s = np.sin(theta / 2)
        phase_m = np.exp(-0.5j * theta)
        phase_p = np.exp(0.5j * theta)
    a, b = states[:, zero], states[:, one]
    out = states.copy()
    if axis == "rx":
        out[:, zero] = c * a - 1j * s * b
        out[:, one] = -1j * s * a + c * b
    elif axis == "ry":
        out[:, zero] = c * a - s * b
        out[:, one] = s * a + c * b
    elif axis == "rz":
        out[:, zero] = phase_m * a
        out[:, one] = phase_p * b
    else:
        raise ValueError(axis)
    return out


def apply_pauli(states, axis, qubit):
    n_qubits = int(round(math.log2(states.shape[1])))
    zero, one = basis_pairs(n_qubits, qubit)
    a, b = states[:, zero], states[:, one]
    out = states.copy()
    if axis == "x":
        out[:, zero], out[:, one] = b, a
    elif axis == "y":
        out[:, zero], out[:, one] = -1j * b, 1j * a
    elif axis == "z":
        out[:, zero], out[:, one] = a, -b
    else:
        raise ValueError(axis)
    return out


def apply_cnot(states):
    return states[:, [0, 1, 3, 2]]


def expectation_z0(states):
    n_qubits = int(round(math.log2(states.shape[1])))
    zero, one = basis_pairs(n_qubits, 0)
    return np.sum(np.abs(states[:, zero]) ** 2, axis=1) - np.sum(np.abs(states[:, one]) ** 2, axis=1)


def forward_with_tape(X, theta, layout, layers, input_scale=1.0, entangler=True, deep_fixed3=False):
    X = np.asarray(X, dtype=float)
    n_qubits, schedule, n_params = build_schedule(layout, layers, entangler, deep_fixed3)
    if len(theta) != n_params:
        raise ValueError(f"Expected {n_params} parameters, received {len(theta)}")
    states = np.zeros((len(X), 2 ** n_qubits), dtype=complex)
    states[:, 0] = 1.0
    tape = []
    for gate, source, qubit, p_index in schedule:
        if gate == "cx":
            states = apply_cnot(states)
            tape.append((gate, None, None, None, None))
            continue
        if source == "x1":
            angle = input_scale * X[:, 0]
        elif source == "x2":
            angle = input_scale * X[:, 1]
        else:
            angle = float(theta[p_index])
        states = apply_rotation(states, gate, angle, qubit)
        tape.append((gate, angle, qubit, p_index, states.copy() if p_index is not None else None))
    return expectation_z0(states), states, tape, n_qubits


def predict_qnn(X, theta, layout, layers, input_scale=1.0, entangler=True, deep_fixed3=False):
    pred, _, _, _ = forward_with_tape(X, theta, layout, layers, input_scale, entangler, deep_fixed3)
    return pred


def loss_and_gradient(X, y, theta, layout, layers, input_scale=1.0, entangler=True, deep_fixed3=False):
    y = np.asarray(y, dtype=float).reshape(-1)
    pred, final_state, tape, _ = forward_with_tape(X, theta, layout, layers, input_scale, entangler, deep_fixed3)
    residual = pred - y
    loss = float(np.mean(residual ** 2))
    gradient = np.zeros_like(theta, dtype=float)
    adjoint = apply_pauli(final_state, "z", 0)
    for gate, angle, qubit, p_index, state_after in reversed(tape):
        if gate == "cx":
            adjoint = apply_cnot(adjoint)
            continue
        if p_index is not None:
            dstate = -0.5j * apply_pauli(state_after, gate[-1], qubit)
            derivative = 2.0 * np.real(np.sum(np.conj(adjoint) * dstate, axis=1))
            gradient[p_index] += float(np.mean(2.0 * residual * derivative))
        adjoint = apply_rotation(adjoint, gate, angle, qubit, inverse=True)
    return loss, gradient, pred


def fit_qnn(X, y, layout, layers, seed, steps=ADAM_STEPS, learning_rate=ADAM_RATE,
            input_scale=1.0, entangler=True, deep_fixed3=False):
    _, _, n_params = build_schedule(layout, layers, entangler, deep_fixed3)
    rng = np.random.default_rng(seed)
    theta = rng.uniform(-np.pi, np.pi, size=n_params)
    m = np.zeros_like(theta)
    v = np.zeros_like(theta)
    for step in range(1, steps + 1):
        loss, grad, _ = loss_and_gradient(X, y, theta, layout, layers, input_scale, entangler, deep_fixed3)
        m = 0.9 * m + 0.1 * grad
        v = 0.999 * v + 0.001 * grad ** 2
        m_hat = m / (1.0 - 0.9 ** step)
        v_hat = v / (1.0 - 0.999 ** step)
        theta -= learning_rate * m_hat / (np.sqrt(v_hat) + 1e-8)
    final_loss, _, final_pred = loss_and_gradient(X, y, theta, layout, layers, input_scale, entangler, deep_fixed3)
    return theta, final_loss, final_pred


# Check the adjoint gradient against central finite differences on a tiny deterministic batch.
check_x = np.asarray([[0.12, -0.31], [0.8, 0.45], [-1.2, 0.6]])
check_y = np.asarray([0.1, -0.3, 0.4])
check_theta = np.asarray([0.23, -0.48, 0.31, 0.17, -0.22, 0.11])
check_loss, check_grad, _ = loss_and_gradient(check_x, check_y, check_theta, "parallel", 3)
eps = 1e-6
numeric = np.zeros_like(check_theta)
for j in range(len(check_theta)):
    plus, minus = check_theta.copy(), check_theta.copy()
    plus[j] += eps
    minus[j] -= eps
    lp = np.mean((predict_qnn(check_x, plus, "parallel", 3) - check_y) ** 2)
    lm = np.mean((predict_qnn(check_x, minus, "parallel", 3) - check_y) ** 2)
    numeric[j] = (lp - lm) / (2 * eps)
gradient_error = float(np.max(np.abs(check_grad - numeric)))
assert gradient_error < 2e-5, f"Adjoint gradient check failed: {gradient_error}"
print(f"Gradient check passed; maximum absolute error={gradient_error:.3g}")


Gradient check passed; maximum absolute error=9.99e-11


## 3. Fourier diagnostics and classical regression baselines

The two-dimensional Fourier map includes all unique sine/cosine pairs with `|k1|, |k2| ≤ 3`, yielding 48 features plus an intercept. This matches the 49 coefficients in the synopsis' 2–12–1 MLP. The spectrum is computed from the normalized 2D DFT. The report tracks the mixed target mode `(1, -1)` and checks support on both the primary and refined grids.


In [4]:
def integer_modes_2d(max_harmonic):
    modes = []
    for k1 in range(-max_harmonic, max_harmonic + 1):
        for k2 in range(-max_harmonic, max_harmonic + 1):
            if (k1, k2) == (0, 0):
                continue
            if k1 < 0 or (k1 == 0 and k2 < 0):
                continue
            modes.append((k1, k2))
    return np.asarray(modes, dtype=float)


def fourier_features_2d(X, max_harmonic):
    modes = integer_modes_2d(max_harmonic)
    phase = np.asarray(X, dtype=float) @ modes.T
    return np.column_stack((np.cos(phase), np.sin(phase)))


def spectrum_metrics_2d(prediction, grid_size, layers):
    coeff = np.fft.fftshift(np.fft.fft2(np.asarray(prediction).reshape(grid_size, grid_size)) / (grid_size ** 2))
    modes = np.rint(np.fft.fftshift(np.fft.fftfreq(grid_size, d=1 / grid_size))).astype(int)
    mag = np.abs(coeff)
    cmax = float(mag.max())
    threshold = max(SPECTRUM_ABS_TOL, SPECTRUM_REL_TOL * cmax)
    active = mag > threshold
    outside = active & ((np.abs(modes[:, None]) > layers) | (np.abs(modes[None, :]) > layers))
    i_x2 = int(np.where(modes == -1)[0][0])
    i_x1 = int(np.where(modes == 1)[0][0])
    return {
        "outside_count": int(outside.sum()),
        "max_outside_magnitude": float(mag[outside].max()) if outside.any() else 0.0,
        "threshold": threshold,
        "cmax": cmax,
        "active_modes": int(active.sum()),
        "max_active_k1": int(np.max(np.abs(modes[np.where(active)[1]]))) if active.any() else 0,
        "max_active_k2": int(np.max(np.abs(modes[np.where(active)[0]]))) if active.any() else 0,
        "mixed_mode_2abs_c_1_minus1": float(2 * mag[i_x2, i_x1]),
        "coefficients": coeff,
        "modes": modes,
    }


fourier2 = LinearRegression().fit(fourier_features_2d(X2_train, 3), y2_train)
mlp2 = MLPRegressor(
    hidden_layer_sizes=(12,), activation="tanh", solver="adam",
    learning_rate_init=0.01, max_iter=4000, tol=1e-7,
    n_iter_no_change=250, random_state=SEED,
)
with warnings.catch_warnings(record=True) as baseline_warnings:
    warnings.simplefilter("always", ConvergenceWarning)
    mlp2.fit(X2_train, y2_train)
pred_fourier2 = fourier2.predict(fourier_features_2d(X2_grid64, 3))
pred_mlp2 = mlp2.predict(X2_grid64)
baseline2_rows = [
    {"task": "2D periodic regression", "model": "Fourier linear |k_j|<=3", "parameters": 49,
     "grid_mse": float(mean_squared_error(y2_grid64.ravel(), pred_fourier2))},
    {"task": "2D periodic regression", "model": "MLP 2-12-1", "parameters": 49,
     "grid_mse": float(mean_squared_error(y2_grid64.ravel(), pred_mlp2))},
]
week1_baseline = pd.read_csv(ROOT / "outputs" / "week1" / "regression_baselines.csv")
pd.DataFrame(baseline2_rows).to_csv(OUT / "week2_2d_classical_baselines.csv", index=False)
print("Fourier features:", fourier_features_2d(X2_train[:1], 3).shape[1], "+ intercept = 49 parameters")
print("MLP parameters:", sum(w.size for w in mlp2.coefs_) + sum(b.size for b in mlp2.intercepts_))
print("MLP convergence warnings:", len(baseline_warnings))
display(pd.DataFrame(baseline2_rows))


Fourier features: 48 + intercept = 49 parameters
MLP parameters: 49
MLP convergence warnings: 0


,task,model,parameters,grid_mse
0,2D periodic regression,Fourier linear |k_j|<=3,49,3.304451e-31
1,2D periodic regression,MLP 2-12-1,49,9.577940e-05


## 4. Full 2D simulator sweep: L=1–6

Both circuit layouts use the same 2,048 training points, dense 64×64 grid, Adam configuration, initialization seeds 0–4, and `2L` trainable parameters. The serial layout uses one qubit; the parallel layout uses two qubits and one CNOT per encoding layer. Each fitted model is evaluated on the 64×64 primary grid and the 128×128 aliasing check. A progress CSV is written after every fit.


In [5]:
sweep_rows = []
spectrum_rows = []
model_parameters = {}
grid_predictions = {}
started_sweep = time.time()
progress_csv = OUT / "week2_sweep_progress.csv"

for layout in ("serial", "parallel"):
    for layers in LAYERS:
        for seed in INIT_SEEDS:
            start = time.time()
            theta, train_mse, _ = fit_qnn(X2_train, y2_train, layout, layers, seed)
            pred64 = predict_qnn(X2_grid64, theta, layout, layers)
            pred128 = predict_qnn(X2_grid128, theta, layout, layers)
            mse64 = float(np.mean((pred64 - y2_grid64.ravel()) ** 2))
            mse128 = float(np.mean((pred128 - y2_grid128.ravel()) ** 2))
            spec64 = spectrum_metrics_2d(pred64, 64, layers)
            spec128 = spectrum_metrics_2d(pred128, 128, layers)
            key = f"{layout}_L{layers}_seed{seed}"
            model_parameters[key] = theta
            grid_predictions[key] = pred64
            row = {
                "layout": layout, "layers": layers, "seed": seed,
                "qubits": 1 if layout == "serial" else 2,
                "trainable_parameters": 2 * layers,
                "train_mse": train_mse,
                "grid64_mse": mse64,
                "grid128_mse": mse128,
                "mixed_mode_2abs_c_1_minus1": spec64["mixed_mode_2abs_c_1_minus1"],
                "support_outside_count_64": spec64["outside_count"],
                "support_outside_count_128": spec128["outside_count"],
                "support_max_outside_64": spec64["max_outside_magnitude"],
                "support_max_outside_128": spec128["max_outside_magnitude"],
                "fit_seconds": time.time() - start,
            }
            sweep_rows.append(row)
            spectrum_rows.extend([
                {"layout": layout, "layers": layers, "seed": seed, "grid_size": 64,
                 "active_modes": spec64["active_modes"], "max_active_k1": spec64["max_active_k1"],
                 "max_active_k2": spec64["max_active_k2"], "outside_count": spec64["outside_count"],
                 "max_outside_magnitude": spec64["max_outside_magnitude"],
                 "threshold": spec64["threshold"],
                 "mixed_mode_2abs_c_1_minus1": spec64["mixed_mode_2abs_c_1_minus1"]},
                {"layout": layout, "layers": layers, "seed": seed, "grid_size": 128,
                 "active_modes": spec128["active_modes"], "max_active_k1": spec128["max_active_k1"],
                 "max_active_k2": spec128["max_active_k2"], "outside_count": spec128["outside_count"],
                 "max_outside_magnitude": spec128["max_outside_magnitude"],
                 "threshold": spec128["threshold"],
                 "mixed_mode_2abs_c_1_minus1": spec128["mixed_mode_2abs_c_1_minus1"]},
            ])
            pd.DataFrame(sweep_rows).to_csv(progress_csv, index=False)
            if seed == 0:
                print(f"{layout:8s} L={layers} seed={seed}: grid64 MSE={mse64:.6g}; grid128 MSE={mse128:.6g}; {row['fit_seconds']:.1f}s", flush=True)

sweep_df = pd.DataFrame(sweep_rows)
spectrum_df = pd.DataFrame(spectrum_rows)
sweep_df.to_csv(OUT / "week2_2d_layer_sweep.csv", index=False)
spectrum_df.to_csv(OUT / "week2_2d_spectrum_support.csv", index=False)
np.savez_compressed(OUT / "week2_2d_predictions_and_parameters.npz", **{
    **{f"theta__{k}": v for k, v in model_parameters.items()},
    **{f"pred64__{k}": v for k, v in grid_predictions.items()},
})

layer_summary = sweep_df.groupby(["layout", "layers"], as_index=False).agg(
    trainable_parameters=("trainable_parameters", "first"),
    grid64_mse_mean=("grid64_mse", "mean"), grid64_mse_sd=("grid64_mse", "std"),
    grid128_mse_mean=("grid128_mse", "mean"), grid128_mse_sd=("grid128_mse", "std"),
    mixed_mode_mean=("mixed_mode_2abs_c_1_minus1", "mean"),
    outside_count_64_total=("support_outside_count_64", "sum"),
    outside_count_128_total=("support_outside_count_128", "sum"),
    seeds=("seed", "count"),
)
layer_summary.to_csv(OUT / "week2_2d_layer_summary.csv", index=False)
print(f"Completed {len(sweep_df)} 2D fits in {(time.time()-started_sweep)/60:.1f} minutes")
display(layer_summary)


serial   L=1 seed=0: grid64 MSE=0.42023; grid128 MSE=0.42023; 6.5s
serial   L=2 seed=0: grid64 MSE=0.331485; grid128 MSE=0.331485; 7.7s
serial   L=3 seed=0: grid64 MSE=0.208525; grid128 MSE=0.208525; 13.0s
serial   L=4 seed=0: grid64 MSE=0.230809; grid128 MSE=0.230809; 18.1s
serial   L=5 seed=0: grid64 MSE=0.134826; grid128 MSE=0.134826; 23.0s
serial   L=6 seed=0: grid64 MSE=0.149897; grid128 MSE=0.149897; 27.2s
parallel L=1 seed=0: grid64 MSE=0.170156; grid128 MSE=0.170156; 6.8s
parallel L=2 seed=0: grid64 MSE=0.170021; grid128 MSE=0.170021; 13.4s
parallel L=3 seed=0: grid64 MSE=0.152985; grid128 MSE=0.152985; 19.6s
parallel L=4 seed=0: grid64 MSE=0.11805; grid128 MSE=0.11805; 26.0s
parallel L=5 seed=0: grid64 MSE=0.0794729; grid128 MSE=0.0794729; 32.7s
parallel L=6 seed=0: grid64 MSE=0.0694252; grid128 MSE=0.0694252; 36.8s
Completed 60 2D fits in 19.2 minutes


,layout,layers,trainable_parameters,grid64_mse_mean,grid64_mse_sd,grid128_mse_mean,grid128_mse_sd,mixed_mode_mean,outside_count_64_total,outside_count_128_total,seeds
0,parallel,1,2,0.170156,1.962616e-17,0.170156,3.399350e-17,2.084058e-18,0,0,5
1,parallel,2,4,0.170021,1.434919e-06,0.170021,1.434919e-06,2.656670e-18,0,0,5
2,parallel,3,6,0.158621,1.260291e-02,0.158621,1.260291e-02,2.364044e-01,0,0,5
3,parallel,4,8,0.116630,3.358240e-03,0.116630,3.358240e-03,2.092231e-01,0,0,5
4,parallel,5,10,0.078790,1.035838e-02,0.078790,1.035838e-02,1.635807e-01,0,0,5
5,parallel,6,12,0.067262,6.110173e-03,0.067262,6.110173e-03,1.766239e-01,0,0,5
6,serial,1,2,0.420326,1.316035e-04,0.420326,1.316035e-04,4.996738e-01,0,0,5
7,serial,2,4,0.258250,4.095190e-02,0.258250,4.095190e-02,4.812496e-02,0,0,5
8,serial,3,6,0.171569,2.482722e-02,0.171569,2.482722e-02,2.866934e-01,0,0,5
9,serial,4,8,0.176958,6.210352e-02,0.176958,6.210352e-02,2.242891e-01,0,0,5


## 5. Depth and entanglement ablations

The depth control keeps exactly three feature encodings and six trainable scalars. It adds a second trainable/entangling block after each encoding; the reference circuit has one block per encoding. This tests added trainable depth while holding the encoding count and total parameter count fixed. A second ablation removes CNOTs from the parallel L=3 model while keeping its width, trainable angles, inputs, and optimizer fixed.


In [6]:
ablation_rows = []
ablation_predictions = {}
ablation_parameters = {}
for condition, layout, deep, entangle in (
    ("parallel_L3_reference", "parallel", False, True),
    ("parallel_L3_deep_fixed3_encodings", "parallel", True, True),
    ("parallel_L3_no_CNOT", "parallel_no_entangler", False, False),
):
    for seed in INIT_SEEDS:
        start = time.time()
        theta, train_mse, _ = fit_qnn(
            X2_train, y2_train, layout, 3, seed,
            entangler=entangle, deep_fixed3=deep,
        )
        pred64 = predict_qnn(
            X2_grid64, theta, layout, 3,
            entangler=entangle, deep_fixed3=deep,
        )
        pred128 = predict_qnn(
            X2_grid128, theta, layout, 3,
            entangler=entangle, deep_fixed3=deep,
        )
        spec64 = spectrum_metrics_2d(pred64, 64, 3)
        spec128 = spectrum_metrics_2d(pred128, 128, 3)
        key = f"{condition}_seed{seed}"
        ablation_parameters[key] = theta
        ablation_predictions[key] = pred64
        ablation_rows.append({
            "condition": condition, "seed": seed, "encodings_per_feature": 3,
            "qubits": 2, "trainable_parameters": 6,
            "train_mse": train_mse,
            "grid64_mse": float(np.mean((pred64 - y2_grid64.ravel()) ** 2)),
            "grid128_mse": float(np.mean((pred128 - y2_grid128.ravel()) ** 2)),
            "mixed_mode_2abs_c_1_minus1": spec64["mixed_mode_2abs_c_1_minus1"],
            "outside_count_64": spec64["outside_count"],
            "outside_count_128": spec128["outside_count"],
            "fit_seconds": time.time() - start,
        })
        print(f"{condition:36s} seed={seed}: grid MSE={ablation_rows[-1]['grid64_mse']:.6g}", flush=True)
ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(OUT / "week2_2d_ablations.csv", index=False)
np.savez_compressed(OUT / "week2_2d_ablation_predictions.npz", **{
    **{f"theta__{k}": v for k, v in ablation_parameters.items()},
    **{f"pred64__{k}": v for k, v in ablation_predictions.items()},
})
ablation_summary = ablation_df.groupby("condition", as_index=False).agg(
    grid64_mse_mean=("grid64_mse", "mean"), grid64_mse_sd=("grid64_mse", "std"),
    grid128_mse_mean=("grid128_mse", "mean"), grid128_mse_sd=("grid128_mse", "std"),
    mixed_mode_mean=("mixed_mode_2abs_c_1_minus1", "mean"),
    outside_count_64_total=("outside_count_64", "sum"),
    outside_count_128_total=("outside_count_128", "sum"), seeds=("seed", "count"),
)
ablation_summary.to_csv(OUT / "week2_2d_ablation_summary.csv", index=False)
display(ablation_summary)


parallel_L3_reference                seed=0: grid MSE=0.152985
parallel_L3_reference                seed=1: grid MSE=0.152984
parallel_L3_reference                seed=2: grid MSE=0.152985
parallel_L3_reference                seed=3: grid MSE=0.152985
parallel_L3_reference                seed=4: grid MSE=0.181166
parallel_L3_deep_fixed3_encodings    seed=0: grid MSE=0.0831114
parallel_L3_deep_fixed3_encodings    seed=1: grid MSE=0.0761445
parallel_L3_deep_fixed3_encodings    seed=2: grid MSE=0.0831104
parallel_L3_deep_fixed3_encodings    seed=3: grid MSE=0.0901371
parallel_L3_deep_fixed3_encodings    seed=4: grid MSE=0.078676
parallel_L3_no_CNOT                  seed=0: grid MSE=0.170369
parallel_L3_no_CNOT                  seed=1: grid MSE=0.170359
parallel_L3_no_CNOT                  seed=2: grid MSE=0.17016
parallel_L3_no_CNOT                  seed=3: grid MSE=0.170538
parallel_L3_no_CNOT                  seed=4: grid MSE=0.170349


,condition,grid64_mse_mean,grid64_mse_sd,grid128_mse_mean,grid128_mse_sd,mixed_mode_mean,outside_count_64_total,outside_count_128_total,seeds
0,parallel_L3_deep_fixed3_encodings,0.082236,0.005332,0.082236,0.005332,6.520403e-02,0,0,5
1,parallel_L3_no_CNOT,0.170355,0.000134,0.170355,0.000134,4.966936e-18,0,0,5
2,parallel_L3_reference,0.158621,0.012603,0.158621,0.012603,2.364044e-01,0,0,5


## 6. Two-spiral classification on the shared folds

Each method uses the same five stratified 1,600/400 training/test splits from Week 1. The Fourier logistic model uses the fixed `|k_j| ≤ 4` feature map. The parameter-matched MLP is 2–1–1 (five trainable weights/biases), close to the six-angle, three-layer QNN. The QNN uses two qubits, three parallel data re-uploads, and a CNOT per layer; spiral coordinates are scaled to `[-π, π]`. No preprocessing statistic is learned from a test fold.


In [7]:
def predict_binary(score):
    return (np.asarray(score) < 0.0).astype(int)


X_spiral_scaled = np.pi * X_spiral
y_spiral_signed = np.where(y_spiral == 0, 1.0, -1.0)
spiral_rows = []
qnn_spiral_parameters = []
with warnings.catch_warnings(record=True) as spiral_warnings:
    warnings.simplefilter("always", ConvergenceWarning)
    for fold_id, (train_idx, test_idx) in enumerate(zip(fold_train, fold_test)):
        Xtr, Xte = X_spiral[train_idx], X_spiral[test_idx]
        # Fixed Fourier map, with its logistic model fitted on this training fold only.
        fourier_clf = LogisticRegression(C=1.0, max_iter=2000, solver="lbfgs", random_state=SEED + fold_id)
        fourier_clf.fit(fourier_features_2d(Xtr, 4), y_spiral[train_idx])
        pred_fourier = fourier_clf.predict(fourier_features_2d(Xte, 4))
        spiral_rows.append({"fold": fold_id, "model": "Fourier logistic |k_j|<=4", "parameters": 81,
                            "accuracy": accuracy_score(y_spiral[test_idx], pred_fourier),
                            "macro_f1": f1_score(y_spiral[test_idx], pred_fourier, average="macro")})

        mlp_clf = make_pipeline(
            StandardScaler(),
            MLPClassifier(hidden_layer_sizes=(1,), activation="tanh", solver="lbfgs",
                          alpha=1e-4, max_iter=1500, random_state=SEED + fold_id),
        )
        mlp_clf.fit(Xtr, y_spiral[train_idx])
        pred_mlp = mlp_clf.predict(Xte)
        spiral_rows.append({"fold": fold_id, "model": "MLP 2-1-1 (parameter matched)", "parameters": 5,
                            "accuracy": accuracy_score(y_spiral[test_idx], pred_mlp),
                            "macro_f1": f1_score(y_spiral[test_idx], pred_mlp, average="macro")})

        theta, train_mse, _ = fit_qnn(
            X_spiral_scaled[train_idx], y_spiral_signed[train_idx],
            "parallel", 3, SEED + fold_id,
            steps=300, learning_rate=0.01, input_scale=1.0,
        )
        qnn_spiral_parameters.append(theta)
        qnn_score = predict_qnn(X_spiral_scaled[test_idx], theta, "parallel", 3)
        pred_qnn = predict_binary(qnn_score)
        spiral_rows.append({"fold": fold_id, "model": "2-qubit re-uploading QNN L=3", "parameters": 6,
                            "accuracy": accuracy_score(y_spiral[test_idx], pred_qnn),
                            "macro_f1": f1_score(y_spiral[test_idx], pred_qnn, average="macro"),
                            "train_mse": train_mse})
        print(f"fold {fold_id+1}/5 complete: QNN accuracy={spiral_rows[-1]['accuracy']:.3f}", flush=True)

spiral_df = pd.DataFrame(spiral_rows)
spiral_df.to_csv(OUT / "week2_spiral_classification_fold_scores.csv", index=False)
np.savez_compressed(OUT / "week2_spiral_qnn_parameters.npz", **{
    f"fold_{i}": value for i, value in enumerate(qnn_spiral_parameters)
})
spiral_summary = spiral_df.groupby(["model", "parameters"], as_index=False).agg(
    accuracy_mean=("accuracy", "mean"), accuracy_sd=("accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
    fixed_folds=("fold", "nunique"),
)
spiral_summary.to_csv(OUT / "week2_spiral_classification_summary.csv", index=False)
print("Captured MLP convergence warnings:", len(spiral_warnings))
display(spiral_summary)


fold 1/5 complete: QNN accuracy=0.477
fold 2/5 complete: QNN accuracy=0.458
fold 3/5 complete: QNN accuracy=0.575
fold 4/5 complete: QNN accuracy=0.583
fold 5/5 complete: QNN accuracy=0.502
Captured MLP convergence warnings: 0


,model,parameters,accuracy_mean,accuracy_sd,macro_f1_mean,macro_f1_sd,fixed_folds
0,2-qubit re-uploading QNN L=3,6,0.519,0.056888,0.518807,0.056859,5
1,Fourier logistic |k_j|<=4,81,0.611,0.030547,0.610810,0.030519,5
2,MLP 2-1-1 (parameter matched),5,0.533,0.042478,0.480839,0.054642,5


## 7. Qiskit Aer depolarizing-noise and finite-shot sweep

The noise study reuses the seed-0 L=3 serial and parallel regression parameters. It evaluates 256 deterministically selected points from the 64×64 grid at 1,024 shots per point, five repeated seeds, and four depolarizing rates. The channel is applied after each one-qubit rotation and each CNOT. The zero-noise row still includes finite-shot measurement sampling.


In [8]:
def make_qiskit_2d_circuit(x, theta, layout, layers=3):
    n_qubits = 1 if layout == "serial" else 2
    qc = QuantumCircuit(n_qubits, 1)
    for layer in range(layers):
        if layout == "serial":
            qc.rx(float(x[0]), 0)
            qc.rz(float(x[1]), 0)
            qc.ry(float(theta[2 * layer]), 0)
            qc.rz(float(theta[2 * layer + 1]), 0)
        else:
            qc.rx(float(x[0]), 0)
            qc.rz(float(x[1]), 1)
            qc.ry(float(theta[2 * layer]), 0)
            qc.rz(float(theta[2 * layer + 1]), 1)
            if layout == "parallel":
                qc.cx(0, 1)
    qc.measure(0, 0)
    return qc


rng_noise = np.random.default_rng(SEED + 901)
eval_indices = np.linspace(0, len(X2_grid64) - 1, NOISE_POINTS, dtype=int)
X_noise = X2_grid64[eval_indices]
y_noise = y2_grid64.ravel()[eval_indices]
noise_rows = []
for layout in ("serial", "parallel"):
    theta = model_parameters[f"{layout}_L3_seed0"]
    noiseless = predict_qnn(X_noise, theta, layout, 3)
    circuits = [make_qiskit_2d_circuit(x, theta, layout, 3) for x in X_noise]
    compiled = transpile(circuits, basis_gates=["rx", "ry", "rz", "cx"], optimization_level=0)
    for p in NOISE_LEVELS:
        noise_model = NoiseModel()
        if p > 0:
            noise_model.add_all_qubit_quantum_error(depolarizing_error(p, 1), ["rx", "ry", "rz"])
            noise_model.add_all_qubit_quantum_error(depolarizing_error(p, 2), ["cx"])
        for repeat in range(NOISE_REPEATS):
            simulator = AerSimulator(method="statevector", noise_model=noise_model,
                                     seed_simulator=SEED + 5000 + repeat)
            result = simulator.run(compiled, shots=NOISE_SHOTS).result()
            counts = result.get_counts()
            if isinstance(counts, dict):
                counts = [counts]
            shot_means = np.asarray([
                (count.get("0", 0) - count.get("1", 0)) / NOISE_SHOTS for count in counts
            ], dtype=float)
            noise_rows.append({
                "layout": layout, "layers": 3, "seed": 0, "depolarizing_probability": p,
                "repeat": repeat, "shots_per_point": NOISE_SHOTS, "evaluation_points": NOISE_POINTS,
                "total_shots": NOISE_SHOTS * NOISE_POINTS,
                "grid_mse_to_target": float(np.mean((shot_means - y_noise) ** 2)),
                "mse_to_noiseless_prediction": float(np.mean((shot_means - noiseless) ** 2)),
                "mean_abs_bias_to_noiseless": float(np.mean(np.abs(shot_means - noiseless))),
            })
            print(f"Aer {layout:8s} p={p:.3g} repeat={repeat}: MSE={noise_rows[-1]['grid_mse_to_target']:.6g}", flush=True)
noise_df = pd.DataFrame(noise_rows)
noise_df.to_csv(OUT / "week2_depolarizing_finite_shot_repeats.csv", index=False)
noise_summary = noise_df.groupby(["layout", "depolarizing_probability"], as_index=False).agg(
    grid_mse_mean=("grid_mse_to_target", "mean"), grid_mse_sd=("grid_mse_to_target", "std"),
    mse_to_noiseless_mean=("mse_to_noiseless_prediction", "mean"),
    mse_to_noiseless_sd=("mse_to_noiseless_prediction", "std"),
    abs_bias_mean=("mean_abs_bias_to_noiseless", "mean"),
    repeats=("repeat", "count"), total_shots_per_repeat=("total_shots", "first"),
)
noise_summary.to_csv(OUT / "week2_depolarizing_finite_shot_summary.csv", index=False)
display(noise_summary)


Aer serial   p=0 repeat=0: MSE=0.305881
Aer serial   p=0 repeat=1: MSE=0.304289
Aer serial   p=0 repeat=2: MSE=0.303579
Aer serial   p=0 repeat=3: MSE=0.308075
Aer serial   p=0 repeat=4: MSE=0.304884
Aer serial   p=0.001 repeat=0: MSE=0.300357
Aer serial   p=0.001 repeat=1: MSE=0.300371
Aer serial   p=0.001 repeat=2: MSE=0.300372
Aer serial   p=0.001 repeat=3: MSE=0.300386
Aer serial   p=0.001 repeat=4: MSE=0.300336
Aer serial   p=0.005 repeat=0: MSE=0.283738
Aer serial   p=0.005 repeat=1: MSE=0.283749
Aer serial   p=0.005 repeat=2: MSE=0.283754
Aer serial   p=0.005 repeat=3: MSE=0.283789
Aer serial   p=0.005 repeat=4: MSE=0.283714
Aer serial   p=0.01 repeat=0: MSE=0.266753
Aer serial   p=0.01 repeat=1: MSE=0.266774
Aer serial   p=0.01 repeat=2: MSE=0.266809
Aer serial   p=0.01 repeat=3: MSE=0.266806
Aer serial   p=0.01 repeat=4: MSE=0.266729
Aer parallel p=0 repeat=0: MSE=0.169871
Aer parallel p=0 repeat=1: MSE=0.168953
Aer parallel p=0 repeat=2: MSE=0.165547
Aer parallel p=0 repeat=3

,layout,depolarizing_probability,grid_mse_mean,grid_mse_sd,mse_to_noiseless_mean,mse_to_noiseless_sd,abs_bias_mean,repeats,total_shots_per_repeat
0,parallel,0.000,0.168169,0.002024,0.000954,0.000068,0.024883,5,262144
1,parallel,0.001,0.170030,0.000038,0.000989,0.000003,0.025846,5,262144
2,parallel,0.005,0.168310,0.000036,0.001104,0.000004,0.026706,5,262144
3,parallel,0.010,0.166406,0.000030,0.001518,0.000005,0.031192,5,262144
4,serial,0.000,0.305342,0.001745,0.000749,0.000078,0.021556,5,262144
5,serial,0.001,0.300365,0.000019,0.000787,0.000003,0.022216,5,262144
6,serial,0.005,0.283749,0.000027,0.001567,0.000003,0.033256,5,262144
7,serial,0.010,0.266774,0.000034,0.003717,0.000006,0.051273,5,262144


## 8. Resource accounting, figures, and saved outputs

Resource counts use the logical circuit before transpilation: one-qubit rotation count, CNOT count, qubit count, trainable parameter count, and logical depth. Week 2 outputs include the complete fit tables, both-grid spectrum checks, classification folds, noise repeats, learned parameters, figures, and a run configuration.


In [9]:
resource_rows = []
for layers in LAYERS:
    resource_rows.append({"layout": "serial", "layers": layers, "qubits": 1,
                          "trainable_parameters": 2 * layers, "encoding_rotations": 2 * layers,
                          "trainable_rotations": 2 * layers, "two_qubit_gates": 0,
                          "logical_depth": 4 * layers})
    resource_rows.append({"layout": "parallel", "layers": layers, "qubits": 2,
                          "trainable_parameters": 2 * layers, "encoding_rotations": 2 * layers,
                          "trainable_rotations": 2 * layers, "two_qubit_gates": layers,
                          "logical_depth": 3 * layers})
resource_rows.extend([
    {"layout": "parallel_L3_no_CNOT", "layers": 3, "qubits": 2,
     "trainable_parameters": 6, "encoding_rotations": 6, "trainable_rotations": 6,
     "two_qubit_gates": 0, "logical_depth": 6},
    {"layout": "parallel_L3_deep_fixed3_encodings", "layers": 3, "qubits": 2,
     "trainable_parameters": 6, "encoding_rotations": 6, "trainable_rotations": 12,
     "two_qubit_gates": 6, "logical_depth": 15},
    {"layout": "parallel_spiral_classifier", "layers": 3, "qubits": 2,
     "trainable_parameters": 6, "encoding_rotations": 6, "trainable_rotations": 6,
     "two_qubit_gates": 3, "logical_depth": 9},
])
resource_df = pd.DataFrame(resource_rows)
resource_df.to_csv(OUT / "week2_circuit_resources.csv", index=False)

# Summarize the existing 1D models with the same reporting columns.
one_d_report = one_d_summary.rename(columns={
    "target": "task", "layers": "layers", "parameters": "trainable_parameters",
    "qnn_grid_mse_mean": "grid_mse_mean", "qnn_grid_mse_sd": "grid_mse_sd",
}).copy()
one_d_report.to_csv(OUT / "week2_reused_1d_summary.csv", index=False)

# Full layer sweep: mean and standard deviation across paired seeds.
fig, ax = plt.subplots(figsize=(8.2, 5.0))
for layout, color in (("serial", "#1769aa"), ("parallel", "#e67e22")):
    sub = layer_summary[layer_summary.layout == layout].sort_values("layers")
    ax.errorbar(sub.layers, sub.grid64_mse_mean, yerr=sub.grid64_mse_sd, marker="o",
                linewidth=2, capsize=4, label=f"{layout} ({1 if layout == 'serial' else 2} qubits)", color=color)
ax.set(yscale="log", xlabel="Encoding layers L", ylabel="64×64 grid MSE",
       title="2D QNN regression across matched-parameter encoding layouts")
ax.set_xticks(LAYERS)
ax.grid(True, which="both", alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG / "week2_2d_layer_sweep.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# 1D full simulator history reused from the Week 1 Advancement sweep.
fig, ax = plt.subplots(figsize=(8.2, 5.0))
for target_name, label, color in (
    ("week1_original", "Original target", "#1769aa"),
    ("high_harmonic", "High-harmonic target", "#bf3f3f"),
):
    sub = one_d_summary[one_d_summary.target == target_name].sort_values("layers")
    ax.errorbar(sub.layers.astype(int), sub.qnn_grid_mse_mean, yerr=sub.qnn_grid_mse_sd,
                marker="o", capsize=4, linewidth=2, label=label, color=color)
ax.set(yscale="log", xlabel="Re-uploading layers L", ylabel="Dense-grid MSE",
       title="1D one-qubit QNN results reused from Week 1 Advancement")
ax.set_xticks(LAYERS)
ax.grid(True, which="both", alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG / "week2_reused_1d_sweep.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# Representative L=3 surfaces use the preregistered seed 0.
serial_surface = grid_predictions["serial_L3_seed0"].reshape(64, 64)
parallel_surface = grid_predictions["parallel_L3_seed0"].reshape(64, 64)
fig, axes = plt.subplots(2, 2, figsize=(10.0, 8.0), constrained_layout=True)
surfaces = [(y2_grid64, "Target"), (serial_surface, "Serial L=3, seed 0"),
            (parallel_surface, "Parallel L=3, seed 0"),
            (parallel_surface - y2_grid64, "Parallel prediction error")]
for ax, (surface, title) in zip(axes.ravel(), surfaces):
    image = ax.imshow(surface, origin="lower", extent=(-np.pi, np.pi, -np.pi, np.pi),
                      aspect="equal", cmap="coolwarm")
    ax.set(title=title, xlabel="x1", ylabel="x2")
    fig.colorbar(image, ax=ax, shrink=0.78)
fig.suptitle("2D target and representative simulator fits", fontsize=15)
fig.savefig(FIG / "week2_2d_surface_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# Spectrum comparison for the target and paired L=3 fits.
specs = [
    spectrum_metrics_2d(y2_grid64.ravel(), 64, 3),
    spectrum_metrics_2d(serial_surface.ravel(), 64, 3),
    spectrum_metrics_2d(parallel_surface.ravel(), 64, 3),
]
fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.3), constrained_layout=True)
for ax, spec, title in zip(axes, specs, ("Target", "Serial L=3", "Parallel L=3")):
    image = ax.imshow(np.log10(np.abs(spec["coefficients"]) + 1e-12), origin="lower",
                      extent=(-32, 31, -32, 31), cmap="magma", vmin=-12, vmax=-0.5)
    ax.set(xlim=(-6, 6), ylim=(-6, 6), xlabel="k1", ylabel="k2", title=title)
    ax.axvline(-3.5, color="cyan", linestyle="--", linewidth=0.8)
    ax.axvline(3.5, color="cyan", linestyle="--", linewidth=0.8)
    ax.axhline(-3.5, color="cyan", linestyle="--", linewidth=0.8)
    ax.axhline(3.5, color="cyan", linestyle="--", linewidth=0.8)
fig.colorbar(image, ax=axes, shrink=0.8, label="log10 |normalized DFT coefficient|")
fig.suptitle("Fourier spectra and candidate support |kj| ≤ 3", fontsize=15)
fig.savefig(FIG / "week2_2d_fourier_spectra.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8.2, 5.0))
for layout, color in (("serial", "#1769aa"), ("parallel", "#e67e22")):
    sub = noise_summary[noise_summary.layout == layout].sort_values("depolarizing_probability")
    ax.errorbar(sub.depolarizing_probability, sub.grid_mse_mean, yerr=sub.grid_mse_sd,
                marker="o", capsize=4, linewidth=2, label=layout, color=color)
ax.set(xlabel="Depolarizing probability per gate", ylabel="MSE on 256-point grid subset",
       title=f"Aer noise and finite-shot sweep ({NOISE_SHOTS} shots/point)")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG / "week2_noise_robustness.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

ablation_order = [
    "parallel_L3_reference",
    "parallel_L3_deep_fixed3_encodings",
    "parallel_L3_no_CNOT",
]
ablation_labels = [
    "Reference\n(3 encodings)",
    "Deeper\n(3 encodings,\n6 parameters)",
    "No CNOT\n(3 encodings,\n6 parameters)",
]
ablation_plot_summary = (
    ablation_df.groupby("condition").grid64_mse.agg(["mean", "std"])
    .reindex(ablation_order)
)
fig, ax = plt.subplots(figsize=(8.5, 4.8))
x = np.arange(len(ablation_order))
colors = ["#1769aa", "#bf3f3f", "#6d9f71"]
ax.bar(
    x, ablation_plot_summary["mean"], yerr=ablation_plot_summary["std"],
    capsize=5, color=colors, alpha=0.88, width=0.62,
    edgecolor="white", linewidth=0.8,
)
for i, condition in enumerate(ablation_order):
    values = ablation_df.loc[ablation_df.condition == condition, "grid64_mse"].to_numpy()
    jitter = np.linspace(-0.10, 0.10, len(values))
    ax.scatter(i + jitter, values, color="#263238", s=24, zorder=3, alpha=0.8)
ax.set(
    xticks=x, xticklabels=ablation_labels, yscale="log",
    ylabel="64×64 grid MSE",
    title="L=3 depth and entangler ablations (mean ± SD)",
)
ax.tick_params(axis="x", labelsize=9)
ax.grid(True, axis="y", which="both", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG / "week2_2d_ablations.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
x = np.arange(len(spiral_summary))
ax.bar(x, spiral_summary.accuracy_mean, yerr=spiral_summary.accuracy_sd,
       capsize=4, color=["#1769aa", "#e67e22", "#6d9f71"][:len(x)])
ax.set(xticks=x, xticklabels=spiral_summary.model, ylim=(0, 1), ylabel="Five-fold accuracy",
       title="Two-spiral classification (mean ± SD)")
ax.tick_params(axis="x", rotation=15)
ax.grid(True, axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG / "week2_spiral_classification.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# Example decision surfaces, fit on fold 0 training data only.
mesh_axis = np.linspace(-1.15, 1.15, 100)
mesh_x, mesh_y = np.meshgrid(mesh_axis, mesh_axis)
mesh = np.column_stack((mesh_x.ravel(), mesh_y.ravel()))
fold0_train = fold_train[0]
fourier_fold0 = LogisticRegression(C=1.0, max_iter=2000, solver="lbfgs", random_state=SEED)
fourier_fold0.fit(fourier_features_2d(X_spiral[fold0_train], 4), y_spiral[fold0_train])
decision_fourier = fourier_fold0.decision_function(fourier_features_2d(mesh, 4))
theta_fold0 = qnn_spiral_parameters[0]
decision_qnn = predict_qnn(np.pi * mesh, theta_fold0, "parallel", 3)
fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.6), constrained_layout=True)
for ax, values, title in zip(axes, (decision_fourier, decision_qnn),
                             ("Fourier logistic, fold 1 fit", "QNN, fold 1 fit")):
    ax.contourf(mesh_x, mesh_y, values.reshape(mesh_x.shape), levels=[-10, 0, 10],
                colors=["#f2a7a0", "#9ecae1"], alpha=0.65)
    for label in (0, 1):
        mask = (y_spiral[fold0_train] == label)
        ax.scatter(X_spiral[fold0_train][mask, 0], X_spiral[fold0_train][mask, 1],
                   s=4, alpha=0.45, label=f"class {label}")
    ax.set(title=title, xlabel="x1", ylabel="x2", xlim=(-1.15, 1.15), ylim=(-1.15, 1.15), aspect="equal")
axes[0].legend(frameon=False, markerscale=3)
fig.suptitle("Two-spiral decision boundaries (fold-local fits)", fontsize=15)
fig.savefig(FIG / "week2_spiral_decision_boundaries.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

run_config = {
    "milestone": "Week 2 simulator results",
    "student": "Rana Muhammad Hamza",
    "roll_number": "25280103",
    "research_question": "How do encoding repetitions and layer count set the frequencies a QNN can represent?",
    "seed_data": SEED,
    "initialization_seeds": list(INIT_SEEDS),
    "layers": list(LAYERS),
    "training": {"optimizer": "Adam", "learning_rate": ADAM_RATE, "maximum_updates": ADAM_STEPS,
                 "gradient": "exact statevector adjoint", "loss": "full-batch MSE"},
    "2d_layouts": {
        "serial": {"qubits": 1, "encoding": "RX(x1), RZ(x2) on one qubit", "trainable_parameters_per_layer": 2},
        "parallel": {"qubits": 2, "encoding": "RX(x1) on q0; RZ(x2) on q1; one CNOT per layer",
                     "trainable_parameters_per_layer": 2},
    },
    "spectrum": {"primary_grid": [64, 64], "refinement_grid": [128, 128],
                 "absolute_threshold": SPECTRUM_ABS_TOL, "relative_threshold_of_cmax": SPECTRUM_REL_TOL},
    "classification": {"dataset": "saved balanced two spirals", "examples": int(len(y_spiral)),
                       "five_saved_stratified_folds": True, "qnn_layout": "parallel two-qubit L=3",
                       "qnn_updates_per_fold": 300, "scaling": "pi times coordinates"},
    "noise": {"backend": "Qiskit Aer statevector with stochastic depolarizing channels",
              "probabilities_per_gate": list(NOISE_LEVELS), "shots_per_point": NOISE_SHOTS,
              "points_per_repeat": NOISE_POINTS, "repeats": NOISE_REPEATS,
              "single_qubit_gates": ["rx", "ry", "rz"], "two_qubit_gate": "cx"},
    "package_versions": {p: metadata.version(p) for p in ("numpy", "pandas", "matplotlib", "scikit-learn", "qiskit", "qiskit-aer")},
    "hardware_execution": False,
}
(OUT / "week2_run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

results_readme = (
    "# Week 2 Simulator Results Bundle\n\n"
    "**Student:** Rana Muhammad Hamza  \n**Roll number:** 25280103  \n"
    "**Milestone:** Week 2, full simulator results / noise / ablations\n\n"
    "## Notebook\n\n"
    "Run `notebooks/25280103_Week2_Simulator_Results.ipynb` from the project root with the **Python (QNN Fourier Spectrum)** kernel. "
    "It reads the saved Week 1 data and fixed folds, reuses the completed Week 1 Advancement 1D sweep, and writes all Week 2 outputs into this folder.\n\n"
    "## Experiment coverage\n\n"
    "- 2D periodic regression, serial one-qubit and parallel two-qubit encoding, L=1-6, five paired seeds, 1,000 full-batch Adam updates per fit.\n"
    "- DFT support checks on 64x64 and refined 128x128 grids using `max(1e-6, 0.01*c_max)`.\n"
    "- L=3 depth control with exactly three feature encodings and six trainable parameters; CNOT removal ablation.\n"
    "- Two-spiral classification with the saved five folds, a Fourier logistic model, a five-parameter MLP, and a two-qubit re-uploading QNN.\n"
    "- Qiskit Aer depolarizing noise after one-qubit rotations and CNOTs, with 1,024 shots per point and five repeats at four noise rates.\n\n"
    "## Files\n\n"
    "- `week2_2d_layer_sweep.csv`, `week2_2d_layer_summary.csv`, `week2_2d_spectrum_support.csv`\n"
    "- `week2_2d_ablations.csv`, `week2_2d_ablation_summary.csv`, `week2_circuit_resources.csv`\n"
    "- `week2_spiral_classification_fold_scores.csv`, `week2_spiral_classification_summary.csv`\n"
    "- `week2_depolarizing_finite_shot_repeats.csv`, `week2_depolarizing_finite_shot_summary.csv`\n"
    "- `week2_run_config.json`, trained parameters and prediction NPZ files, `figures/`\n"
    "- Results draft: `report/Week2_Results_Draft_25280103.pdf`\n\n"
    "All quantum data in this bundle comes from statevector or Qiskit Aer simulation. The hardware evaluation is scheduled for Week 3.\n"
)
(OUT / "README.md").write_text(results_readme, encoding="utf-8")

print("Saved Week 2 CSV tables, trained parameters, configuration, README, and figures to", OUT)
print("2D sweep fits:", len(sweep_df), "| spectrum checks:", len(spectrum_df))
print("Depth/entanglement ablation fits:", len(ablation_df), "| spiral fold rows:", len(spiral_df))
print("Noisy Aer circuit batches:", len(noise_df), "| shots per batch:", NOISE_SHOTS * NOISE_POINTS)


Saved Week 2 CSV tables, trained parameters, configuration, README, and figures to C:\Users\Teacher\Desktop\qml\qnn-fourier-spectrum\outputs\week2
2D sweep fits: 60 | spectrum checks: 120
Depth/entanglement ablation fits: 15 | spiral fold rows: 15
Noisy Aer circuit batches: 40 | shots per batch: 262144


<ipython-input-1-41657fe4dc79>:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
<ipython-input-1-41657fe4dc79>:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
<ipython-input-1-41657fe4dc79>:81: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
<ipython-input-1-41657fe4dc79>:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
<ipython-input-1-41657fe4dc79>:116: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
<ipython-input-1-41657fe4dc79>:132: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
<ipython-input-1-41657fe4dc79>:145: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
<ipython-input-1-41657fe4dc79>:171: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Results interpretation and Week 3 handoff

The following tables are computed from the saved Week 2 outputs. Interpret the measured support against the declared tolerance and both grid resolutions. A valid frequency bound means the circuit could represent those modes; it does not guarantee that training reaches the target coefficients. The classification results use shared folds and are reported as mean ± sample standard deviation. Week 3 adds the hardware comparison, subject to backend access.


In [10]:
print("2D matched-layout summary:")
display(layer_summary)
print("L=3 depth and entangler ablation:")
display(ablation_summary)
print("Five-fold spiral classification:")
display(spiral_summary)
print("Aer noise / finite-shot summary:")
display(noise_summary)
print("Resource summary:")
display(resource_df)


2D matched-layout summary:
L=3 depth and entangler ablation:
Five-fold spiral classification:
Aer noise / finite-shot summary:
Resource summary:


,layout,layers,trainable_parameters,grid64_mse_mean,grid64_mse_sd,grid128_mse_mean,grid128_mse_sd,mixed_mode_mean,outside_count_64_total,outside_count_128_total,seeds
0,parallel,1,2,0.170156,1.962616e-17,0.170156,3.399350e-17,2.084058e-18,0,0,5
1,parallel,2,4,0.170021,1.434919e-06,0.170021,1.434919e-06,2.656670e-18,0,0,5
2,parallel,3,6,0.158621,1.260291e-02,0.158621,1.260291e-02,2.364044e-01,0,0,5
3,parallel,4,8,0.116630,3.358240e-03,0.116630,3.358240e-03,2.092231e-01,0,0,5
4,parallel,5,10,0.078790,1.035838e-02,0.078790,1.035838e-02,1.635807e-01,0,0,5
5,parallel,6,12,0.067262,6.110173e-03,0.067262,6.110173e-03,1.766239e-01,0,0,5
6,serial,1,2,0.420326,1.316035e-04,0.420326,1.316035e-04,4.996738e-01,0,0,5
7,serial,2,4,0.258250,4.095190e-02,0.258250,4.095190e-02,4.812496e-02,0,0,5
8,serial,3,6,0.171569,2.482722e-02,0.171569,2.482722e-02,2.866934e-01,0,0,5
9,serial,4,8,0.176958,6.210352e-02,0.176958,6.210352e-02,2.242891e-01,0,0,5


,condition,grid64_mse_mean,grid64_mse_sd,grid128_mse_mean,grid128_mse_sd,mixed_mode_mean,outside_count_64_total,outside_count_128_total,seeds
0,parallel_L3_deep_fixed3_encodings,0.082236,0.005332,0.082236,0.005332,6.520403e-02,0,0,5
1,parallel_L3_no_CNOT,0.170355,0.000134,0.170355,0.000134,4.966936e-18,0,0,5
2,parallel_L3_reference,0.158621,0.012603,0.158621,0.012603,2.364044e-01,0,0,5


,model,parameters,accuracy_mean,accuracy_sd,macro_f1_mean,macro_f1_sd,fixed_folds
0,2-qubit re-uploading QNN L=3,6,0.519,0.056888,0.518807,0.056859,5
1,Fourier logistic |k_j|<=4,81,0.611,0.030547,0.610810,0.030519,5
2,MLP 2-1-1 (parameter matched),5,0.533,0.042478,0.480839,0.054642,5


,layout,depolarizing_probability,grid_mse_mean,grid_mse_sd,mse_to_noiseless_mean,mse_to_noiseless_sd,abs_bias_mean,repeats,total_shots_per_repeat
0,parallel,0.000,0.168169,0.002024,0.000954,0.000068,0.024883,5,262144
1,parallel,0.001,0.170030,0.000038,0.000989,0.000003,0.025846,5,262144
2,parallel,0.005,0.168310,0.000036,0.001104,0.000004,0.026706,5,262144
3,parallel,0.010,0.166406,0.000030,0.001518,0.000005,0.031192,5,262144
4,serial,0.000,0.305342,0.001745,0.000749,0.000078,0.021556,5,262144
5,serial,0.001,0.300365,0.000019,0.000787,0.000003,0.022216,5,262144
6,serial,0.005,0.283749,0.000027,0.001567,0.000003,0.033256,5,262144
7,serial,0.010,0.266774,0.000034,0.003717,0.000006,0.051273,5,262144


,layout,layers,qubits,trainable_parameters,encoding_rotations,trainable_rotations,two_qubit_gates,logical_depth
0,serial,1,1,2,2,2,0,4
1,parallel,1,2,2,2,2,1,3
2,serial,2,1,4,4,4,0,8
3,parallel,2,2,4,4,4,2,6
4,serial,3,1,6,6,6,0,12
5,parallel,3,2,6,6,6,3,9
6,serial,4,1,8,8,8,0,16
7,parallel,4,2,8,8,8,4,12
8,serial,5,1,10,10,10,0,20
9,parallel,5,2,10,10,10,5,15


## Limitations

The two-qubit models are classically simulable at this width; the work makes no quantum-advantage claim. The depth ablation changes entangling-block count and angle sharing while keeping the total number of trainable scalars at six, so its result is an architectural comparison rather than a universal depth law. The noise study uses a depolarizing channel and finite shots in Aer; it does not model device calibration drift, crosstalk, readout asymmetry, or transpilation to a hardware backend. The spiral QNN is a compact three-layer model evaluated on five saved folds; a weak result is evidence about this circuit/training setup rather than all quantum classifiers.

**Week 3 plan:** confirm backend availability; freeze a trained one-qubit L=3 regressor; evaluate 64 periodic points with 1,024 hardware shots; compare hardware, finite-shot Aer, and noiseless curves; record backend, calibration date, transpiled depth, and shot counts.
